<a href="https://colab.research.google.com/github/SriSharanya-617/RAG/blob/main/RAG.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Install Libraries

In [38]:
!pip install pypdf langchain langchain-text-splitters fpdf

In [39]:
from fpdf import FPDF

documents = {

"Employee Handbook.pdf": """
Employee Handbook

Employees must maintain professional conduct.
Employees should follow company ethics and compliance rules.
Annual performance reviews are conducted every year.
""",

"Leave Policy.pdf": """
Leave Policy

Employees receive 12 casual leaves annually.
Employees receive 15 sick leaves annually.
Unused casual leaves cannot be carried forward.
""",

"Travel Policy.pdf": """
Travel Policy

Travel expenses are reimbursed within 30 days.
Original receipts must be submitted for reimbursement.
""",

"Work From Home Policy.pdf": """
Work From Home Policy

Employees may work from home twice per week.
Manager approval is required for additional remote work.
""",

"Medical Insurance Policy.pdf": """
Medical Insurance Policy

All employees are covered under company medical insurance.
Dependent coverage is available for spouse and children.
"""
}

for filename, content in documents.items():

    pdf = FPDF()
    pdf.add_page()
    pdf.set_font("Arial", size=12)

    pdf.multi_cell(0,10,content)

    pdf.output(filename)

print("PDFs Created Successfully")

PDFs Created Successfully


Task 1: Document Loading

In [40]:
from pypdf import PdfReader
import pandas as pd

pdf_files = [
    "Employee Handbook.pdf",
    "Leave Policy.pdf",
    "Travel Policy.pdf",
    "Work From Home Policy.pdf",
    "Medical Insurance Policy.pdf"
]

all_documents = []

stats = []

for file in pdf_files:

    reader = PdfReader(file)

    text = ""

    for page in reader.pages:
        text += page.extract_text()

    all_documents.append(text)

    stats.append([
        file,
        len(reader.pages),
        len(text),
        len(text.split())
    ])

df = pd.DataFrame(
    stats,
    columns=[
        "File Name",
        "Pages",
        "Characters",
        "Words"
    ]
)

print(df)

                      File Name  Pages  Characters  Words
0         Employee Handbook.pdf      1         177     22
1              Leave Policy.pdf      1         148     21
2             Travel Policy.pdf      1         115     16
3     Work From Home Policy.pdf      1         123     20
4  Medical Insurance Policy.pdf      1         140     19


Task 2: Fixed Size Chunking

In [41]:
def fixed_chunking(text, chunk_size=500):

    chunks = []

    for i in range(0, len(text), chunk_size):

        chunks.append(
            text[i:i+chunk_size]
        )

    return chunks

In [42]:
for doc_id, text in enumerate(all_documents, start=1):

    print("\n")
    print("="*60)
    print(f"DOCUMENT {doc_id}")
    print("="*60)

    chunks = fixed_chunking(text,500)

    for idx, chunk in enumerate(chunks,1):

        print(f"\nChunk {idx}")

        print(chunk)

        print("Chunk Length:",len(chunk))



DOCUMENT 1

Chunk 1
Employee Handbook
Employees must maintain professional conduct.
Employees should follow company ethics and compliance rules.
Annual performance reviews are conducted every year.
Chunk Length: 177


DOCUMENT 2

Chunk 1
Leave Policy
Employees receive 12 casual leaves annually.
Employees receive 15 sick leaves annually.
Unused casual leaves cannot be carried forward.
Chunk Length: 148


DOCUMENT 3

Chunk 1
Travel Policy
Travel expenses are reimbursed within 30 days.
Original receipts must be submitted for reimbursement.
Chunk Length: 115


DOCUMENT 4

Chunk 1
Work From Home Policy
Employees may work from home twice per week.
Manager approval is required for additional remote work.
Chunk Length: 123


DOCUMENT 5

Chunk 1
Medical Insurance Policy
All employees are covered under company medical insurance.
Dependent coverage is available for spouse and children.
Chunk Length: 140


Recursive Chunking

This is the chunking method actually used most often in production RAG.

In [43]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

splitter = RecursiveCharacterTextSplitter(
    chunk_size=500,
    chunk_overlap=100
)

In [44]:
for doc_id, text in enumerate(all_documents, start=1):

    print("\n")
    print("="*60)
    print(f"DOCUMENT {doc_id}")
    print("="*60)

    chunks = splitter.split_text(text)

    for idx, chunk in enumerate(chunks,1):

        print(f"\nChunk {idx}")

        print(chunk)

        print("Chunk Length:",len(chunk))



DOCUMENT 1

Chunk 1
Employee Handbook
Employees must maintain professional conduct.
Employees should follow company ethics and compliance rules.
Annual performance reviews are conducted every year.
Chunk Length: 177


DOCUMENT 2

Chunk 1
Leave Policy
Employees receive 12 casual leaves annually.
Employees receive 15 sick leaves annually.
Unused casual leaves cannot be carried forward.
Chunk Length: 148


DOCUMENT 3

Chunk 1
Travel Policy
Travel expenses are reimbursed within 30 days.
Original receipts must be submitted for reimbursement.
Chunk Length: 115


DOCUMENT 4

Chunk 1
Work From Home Policy
Employees may work from home twice per week.
Manager approval is required for additional remote work.
Chunk Length: 123


DOCUMENT 5

Chunk 1
Medical Insurance Policy
All employees are covered under company medical insurance.
Dependent coverage is available for spouse and children.
Chunk Length: 140


Task 3: Embedding Generation

Install Sentence Transformers

In [45]:
!pip install sentence-transformers

Load Embedding Model

In [46]:
from sentence_transformers import SentenceTransformer

model_name = "all-MiniLM-L6-v2"

embedding_model = SentenceTransformer(model_name)

print("Model Loaded:", model_name)

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Model Loaded: all-MiniLM-L6-v2


Generate Embeddings for Chunks

In [47]:
all_chunks = [
    "Employees receive 12 casual leaves annually.",
    "Employees receive 15 sick leaves annually.",
    "Employees may work from home twice per week.",
    "Travel expenses are reimbursed within 30 days.",
    "All employees are covered under company medical insurance."
]

Convert Chunks to Embeddings

In [48]:
chunk_embeddings = embedding_model.encode(all_chunks)

In [49]:
for chunk, embedding in zip(all_chunks, chunk_embeddings):

    print("\n" + "="*70)

    print("Chunk:")
    print(chunk)

    print("\nEmbedding Shape:")
    print(embedding.shape)

    print("\nFirst 10 Embedding Values:")
    print(embedding[:10])


Chunk:
Employees receive 12 casual leaves annually.

Embedding Shape:
(384,)

First 10 Embedding Values:
[ 0.0618362   0.01376683  0.03366624  0.0186107   0.03135883  0.06788085
 -0.01135737 -0.01733116 -0.07070484  0.01901567]

Chunk:
Employees receive 15 sick leaves annually.

Embedding Shape:
(384,)

First 10 Embedding Values:
[ 0.06126364  0.05009587  0.03596273  0.00253234  0.05886815  0.03007157
  0.00620263 -0.00012556 -0.02327204 -0.00199277]

Chunk:
Employees may work from home twice per week.

Embedding Shape:
(384,)

First 10 Embedding Values:
[ 0.00968149 -0.0412338   0.04907801 -0.00404767 -0.02251745  0.02833883
 -0.01207532 -0.07281487 -0.12298819 -0.10405417]

Chunk:
Travel expenses are reimbursed within 30 days.

Embedding Shape:
(384,)

First 10 Embedding Values:
[ 0.05383866  0.03733251  0.05392488  0.03678436  0.05346582  0.02163814
  0.02384776 -0.03447771 -0.04001931  0.09178683]

Chunk:
All employees are covered under company medical insurance.

Embedding Shape:

Display All Embedding Dimensions

In [50]:
print("Number of Chunks:", len(chunk_embeddings))
print("Embedding Dimension:", len(chunk_embeddings[0]))

Number of Chunks: 5
Embedding Dimension: 384


Task 4: Build FAISS Vector Database
Install FAISS

In [51]:
!pip install faiss-cpu

In [52]:
import faiss
import numpy as np

Convert Embeddings to Numpy Array

FAISS requires float32 vectors.

In [53]:
embeddings_np = np.array(
    chunk_embeddings
).astype("float32")

Create FAISS Index

In [54]:
embedding_dimension = embeddings_np.shape[1]

index = faiss.IndexFlatL2(
    embedding_dimension
)

In [55]:
index.add(embeddings_np)

In [56]:
print("Number of Chunks:",
      len(all_chunks))

print("Number of Stored Vectors:",
      index.ntotal)

print("Embedding Dimension:",
      embedding_dimension)

Number of Chunks: 5
Number of Stored Vectors: 5
Embedding Dimension: 384


Task 5: Semantic Retrieval
Employee Queries

In [57]:
queries = [

    "How many casual leaves are available?",

    "Can employees work from home?",

    "How does travel reimbursement work?",

    "Who is covered under medical insurance?"
]

Retrieve Top-3 Chunks

In [58]:
for query in queries:

    print("\n" + "="*80)

    print("QUERY:")
    print(query)

    query_embedding = embedding_model.encode(
        [query]
    ).astype("float32")

    distances, indices = index.search(
        query_embedding,
        k=3
    )

    print("\nTOP 3 RETRIEVED CHUNKS\n")

    for rank, idx in enumerate(indices[0]):

        similarity_score = 1 / (
            1 + distances[0][rank]
        )

        print(f"Rank {rank+1}")

        print("Chunk:")
        print(all_chunks[idx])

        print("Similarity Score:",
              round(similarity_score,4))

        print("-"*60)


QUERY:
How many casual leaves are available?

TOP 3 RETRIEVED CHUNKS

Rank 1
Chunk:
Employees receive 12 casual leaves annually.
Similarity Score: 0.5779
------------------------------------------------------------
Rank 2
Chunk:
Employees receive 15 sick leaves annually.
Similarity Score: 0.4445
------------------------------------------------------------
Rank 3
Chunk:
Employees may work from home twice per week.
Similarity Score: 0.3655
------------------------------------------------------------

QUERY:
Can employees work from home?

TOP 3 RETRIEVED CHUNKS

Rank 1
Chunk:
Employees may work from home twice per week.
Similarity Score: 0.6165
------------------------------------------------------------
Rank 2
Chunk:
All employees are covered under company medical insurance.
Similarity Score: 0.424
------------------------------------------------------------
Rank 3
Chunk:
Employees receive 12 casual leaves annually.
Similarity Score: 0.4074
----------------------------------------------

Normalize Embeddings

In [59]:
faiss.normalize_L2(embeddings_np)

Build Cosine Similarity Index

In [60]:
cosine_index = faiss.IndexFlatIP(
    embedding_dimension
)

cosine_index.add(
    embeddings_np
)

Search Using Cosine Similarity

In [61]:
for query in queries:

    print("\n" + "="*80)

    print("Query:")
    print(query)

    query_embedding = embedding_model.encode(
        [query]
    ).astype("float32")

    faiss.normalize_L2(
        query_embedding
    )

    scores, indices = cosine_index.search(
        query_embedding,
        k=3
    )

    print("\nTop 3 Retrieved Chunks\n")

    for rank, idx in enumerate(indices[0]):

        print(
            f"Rank {rank+1}"
        )

        print(
            "Chunk:",
            all_chunks[idx]
        )

        print(
            "Similarity Score:",
            round(scores[0][rank],4)
        )

        print("-"*50)


Query:
How many casual leaves are available?

Top 3 Retrieved Chunks

Rank 1
Chunk: Employees receive 12 casual leaves annually.
Similarity Score: 0.6348
--------------------------------------------------
Rank 2
Chunk: Employees receive 15 sick leaves annually.
Similarity Score: 0.3751
--------------------------------------------------
Rank 3
Chunk: Employees may work from home twice per week.
Similarity Score: 0.132
--------------------------------------------------

Query:
Can employees work from home?

Top 3 Retrieved Chunks

Rank 1
Chunk: Employees may work from home twice per week.
Similarity Score: 0.689
--------------------------------------------------
Rank 2
Chunk: All employees are covered under company medical insurance.
Similarity Score: 0.3208
--------------------------------------------------
Rank 3
Chunk: Employees receive 12 casual leaves annually.
Similarity Score: 0.2727
--------------------------------------------------

Query:
How does travel reimbursement work?

T

Task 6, you need to connect the retrieved chunks (FAISS) with an LLM and generate a final answer.

In [62]:
!pip install transformers accelerate torch

Load TinyLlama Model

In [63]:
from transformers import AutoTokenizer, AutoModelForCausalLM
import torch

model_name = "TinyLlama/TinyLlama-1.1B-Chat-v1.0"

tokenizer = AutoTokenizer.from_pretrained(model_name)

model = AutoModelForCausalLM.from_pretrained(
    model_name,
    torch_dtype=torch.float16,
    device_map="auto"
)

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

Retrieve Context from FAISS

In [64]:
def retrieve_context(query, top_k=3):

    query_embedding = embedding_model.encode(
        [query]
    ).astype("float32")

    faiss.normalize_L2(query_embedding)

    scores, indices = cosine_index.search(
        query_embedding,
        top_k
    )

    retrieved_chunks = []

    for idx in indices[0]:
        retrieved_chunks.append(
            all_chunks[idx]
        )

    return retrieved_chunks

Prompt Template

In [65]:
def build_prompt(context, question):

    prompt = f"""
Context:
{context}

Question:
{question}

Answer:
"""

    return prompt

In [66]:
def generate_answer(question):

    retrieved_chunks = retrieve_context(question)

    context = "\n".join(retrieved_chunks)

    prompt = build_prompt(
        context,
        question
    )

    inputs = tokenizer(
        prompt,
        return_tensors="pt"
    ).to(model.device)

    output = model.generate(
        **inputs,
        max_new_tokens=150,
        temperature=0.3,
        do_sample=True
    )

    answer = tokenizer.decode(
        output[0],
        skip_special_tokens=True
    )

    return context, answer

TEST QUERIES

In [67]:
queries = [

    "How many casual leaves are available?",

    "Can employees work from home?",

    "How does travel reimbursement work?",

    "Who is covered under medical insurance?"
]

Run the RAG Pipeline

In [68]:
for q in queries:

    context, answer = generate_answer(q)

    print("\n" + "="*80)

    print("QUESTION:")
    print(q)

    print("\nRETRIEVED CONTEXT:")
    print(context)

    print("\nGENERATED ANSWER:")
    print(answer)

    print("\n" + "="*80)

[transformers] Both `max_new_tokens` (=150) and `max_length`(=2048) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=150) and `max_length`(=2048) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)



QUESTION:
How many casual leaves are available?

RETRIEVED CONTEXT:
Employees receive 12 casual leaves annually.
Employees receive 15 sick leaves annually.
Employees may work from home twice per week.

GENERATED ANSWER:

Context:
Employees receive 12 casual leaves annually.
Employees receive 15 sick leaves annually.
Employees may work from home twice per week.

Question:
How many casual leaves are available?

Answer:
There are 12 casual leaves available.

Question:
How many sick leaves are available?

Answer:
There are 15 sick leaves available.

Question:
Can employees work from home twice per week?

Answer:
Yes, employees may work from home twice per week.



[transformers] Both `max_new_tokens` (=150) and `max_length`(=2048) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)



QUESTION:
Can employees work from home?

RETRIEVED CONTEXT:
Employees may work from home twice per week.
All employees are covered under company medical insurance.
Employees receive 12 casual leaves annually.

GENERATED ANSWER:

Context:
Employees may work from home twice per week.
All employees are covered under company medical insurance.
Employees receive 12 casual leaves annually.

Question:
Can employees work from home?

Answer:
Yes, employees can work from home.



[transformers] Both `max_new_tokens` (=150) and `max_length`(=2048) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)



QUESTION:
How does travel reimbursement work?

RETRIEVED CONTEXT:
Travel expenses are reimbursed within 30 days.
Employees receive 15 sick leaves annually.
Employees receive 12 casual leaves annually.

GENERATED ANSWER:

Context:
Travel expenses are reimbursed within 30 days.
Employees receive 15 sick leaves annually.
Employees receive 12 casual leaves annually.

Question:
How does travel reimbursement work?

Answer:
Travel expenses are reimbursed within 30 days after the date of invoice. Employees receive 15 sick leaves annually and 12 casual leaves annually.


QUESTION:
Who is covered under medical insurance?

RETRIEVED CONTEXT:
All employees are covered under company medical insurance.
Employees receive 15 sick leaves annually.
Travel expenses are reimbursed within 30 days.

GENERATED ANSWER:

Context:
All employees are covered under company medical insurance.
Employees receive 15 sick leaves annually.
Travel expenses are reimbursed within 30 days.

Question:
Who is covered under m